# Ex1

In [5]:
import numpy as np
import pandas as pd

pd.set_option("mode.copy_on_write", True)

results = {}

orig_url = "https://github.com/nickeubank/MIDS_Data/raw/refs/heads/master/mortgages/2004/sample_orig_2004_standard_mortgages.txt.zip"
svcg_url = "https://github.com/nickeubank/MIDS_Data/raw/refs/heads/master/mortgages/2004/sample_svcg_2004_threeyears_standard_mortgages.txt.zip"

orig = pd.read_csv(
    orig_url,
    sep="|",
    compression="zip",
    header=None,
    low_memory=False
)

svcg = pd.read_csv(
    svcg_url,
    sep="|",
    compression="zip",
    header=None,
    low_memory=False,
    dtype={0: "string"}  
)

orig.head()


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,653,200403,Y,203402,20740.0,17,1,P,87,25,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,9,N,9
1,747,200403,N,203402,30700.0,0,2,I,70,34,...,1,Other sellers,Other servicers,NaN,NaN,9,NaN,9,N,9
2,731,200403,N,201902,NaN,0,1,P,80,40,...,2,"PROVIDENT FUNDING ASSOCIATES, L.P.","PROVIDENT FUNDING ASSOCIATES, L.P.",NaN,NaN,9,NaN,9,N,9
3,682,200403,N,201902,NaN,0,1,P,80,30,...,2,Other sellers,Other servicers,NaN,NaN,9,NaN,9,N,9
4,730,200403,N,203402,NaN,0,1,P,80,30,...,1,Other sellers,Other servicers,NaN,NaN,9,NaN,9,N,9


In [4]:
svcg.head()

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,F04Q10000054,200403,126000.0,0,1,359,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,126000.0
1,F04Q10000054,200404,126000.0,0,2,358,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,126000.0
2,F04Q10000054,200405,126000.0,0,3,357,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,126000.0
3,F04Q10000054,200406,126000.0,0,4,356,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,126000.0
4,F04Q10000054,200407,126000.0,0,5,355,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,126000.0


# Ex2

For '''sample_orig_2004_standard_mortgages.txt''', the unit observation is one mortgage (one loan). Each row represents one loan with its fixed characteristics at origination (credit score, loan amount, etc.) For '''sample_svcg_2004_threeyears_standard_mortgages.txt''', the unit of observation is one loan-month. Each row represents one loan in one specific month (payment status, balance, delinquency, etc.)

# Ex3

In [6]:

merged = orig.merge(
    svcg,
    left_on=19,
    right_on=0,
    how="inner",
    validate="1:m"
)


results["ex2_merge_type"] = "1:m"

# Ex4

In [10]:
#print(merged.columns.tolist())

# create delinquency indicator at row level
merged["delinquent_flag"] = ( (pd.to_numeric(merged["3_y"], errors="coerce") > 0) | (merged["3_y"] == "RA")).astype(int)

# collapse to loan level (loan id = column 0 from svcg side)
loan_delinquency = (merged.groupby("0_y")["delinquent_flag"].max().reset_index())

# Ex5

In [13]:
loan_delinquency.columns = ["loan_id", "ever_delinquent"]
# keep one row per mortgage from origination data
orig_cols = [c for c in merged.columns if str(c).endswith("_x")]

mortgage_level = (
    merged[orig_cols]
    .drop_duplicates(subset="19_x")
    .merge(loan_delinquency, left_on="19_x", right_on="loan_id", how="left", validate="1:1")
)

# store results
results["ex5_num_mortgages"] = mortgage_level.shape[0]
results["ex5_share_delinquent"] = round(mortgage_level["ever_delinquent"].mean(), 3)

mortgage_level.head()

,0_x,1_x,2_x,3_x,4_x,5_x,6_x,7_x,8_x,9_x,...,24_x,25_x,26_x,27_x,28_x,29_x,30_x,31_x,loan_id,ever_delinquent
0,653,200403,Y,203402,20740.0,17,1,P,87,25,...,Other servicers,NaN,NaN,9,NaN,9,N,9,F04Q10000054,0
1,747,200403,N,203402,30700.0,0,2,I,70,34,...,Other servicers,NaN,NaN,9,NaN,9,N,9,F04Q10000200,0
2,731,200403,N,201902,NaN,0,1,P,80,40,...,"PROVIDENT FUNDING ASSOCIATES, L.P.",NaN,NaN,9,NaN,9,N,9,F04Q10000236,0
3,682,200403,N,201902,NaN,0,1,P,80,30,...,Other servicers,NaN,NaN,9,NaN,9,N,9,F04Q10000281,0
4,730,200403,N,203402,NaN,0,1,P,80,30,...,Other servicers,NaN,NaN,9,NaN,9,N,9,F04Q10000296,0


# Ex6

In [14]:
# build modeling dataset from mortgage-level data
model_df = mortgage_level[[
    "0_x",   # credit score
    "2_x",   # first time homebuyer flag
    "6_x",   # number of units
    "5_x",   # mortgage insurance percentage
    "7_x",   # occupancy status
    "9_x",   # original DTI
    "10_x",  # original UPB
    "11_x",  # original LTV
    "12_x",  # original interest rate
    "13_x",  # channel
    "14_x",  # prepayment penalty mortgage flag
    "15_x",  # amortization type / product type
    "16_x",  # property state
    "17_x",  # property type
    "21_x",  # original loan term
    "22_x",  # number of borrowers
    "ever_delinquent"
]].copy()

model_df.columns = [
    "credit_score",
    "first_time_homebuyer_flag",
    "num_units",
    "mi_pct",
    "occupancy_status",
    "orig_dti",
    "orig_upb",
    "orig_ltv",
    "orig_interest_rate",
    "channel",
    "ppm_flag",
    "amortization_type",
    "property_state",
    "property_type",
    "orig_loan_term",
    "num_borrowers",
    "ever_delinquent"
]

# treat Freddie Mac missing codes as missing, not as categories
model_df["credit_score"] = pd.to_numeric(model_df["credit_score"], errors="coerce").replace(9999, np.nan)
model_df["first_time_homebuyer_flag"] = model_df["first_time_homebuyer_flag"].replace("9", np.nan)
model_df["num_units"] = pd.to_numeric(model_df["num_units"], errors="coerce").replace(99, np.nan)
model_df["mi_pct"] = pd.to_numeric(model_df["mi_pct"], errors="coerce").replace(999, np.nan)
model_df["occupancy_status"] = model_df["occupancy_status"].replace("9", np.nan)
model_df["orig_dti"] = pd.to_numeric(model_df["orig_dti"], errors="coerce").replace(999, np.nan)
model_df["orig_upb"] = pd.to_numeric(model_df["orig_upb"], errors="coerce")
model_df["orig_ltv"] = pd.to_numeric(model_df["orig_ltv"], errors="coerce").replace(999, np.nan)
model_df["orig_interest_rate"] = pd.to_numeric(model_df["orig_interest_rate"], errors="coerce")
model_df["channel"] = model_df["channel"].replace("9", np.nan)
model_df["ppm_flag"] = model_df["ppm_flag"].replace("9", np.nan)
model_df["amortization_type"] = model_df["amortization_type"].replace("9", np.nan)
model_df["property_state"] = model_df["property_state"].replace("", np.nan)
model_df["property_type"] = model_df["property_type"].replace("99", np.nan)
model_df["orig_loan_term"] = pd.to_numeric(model_df["orig_loan_term"], errors="coerce").replace(999, np.nan)
model_df["num_borrowers"] = pd.to_numeric(model_df["num_borrowers"], errors="coerce").replace(99, np.nan)

model_df.head()

,credit_score,first_time_homebuyer_flag,num_units,mi_pct,occupancy_status,orig_dti,orig_upb,orig_ltv,orig_interest_rate,channel,ppm_flag,amortization_type,property_state,property_type,orig_loan_term,num_borrowers,ever_delinquent
0,653.0,Y,1.0,17.0,P,25.0,127000,87,6.050,R,N,FRM,WI,SF,360,2.0,0
1,747.0,N,2.0,0.0,I,34.0,121000,70,6.125,R,N,FRM,NE,PU,360,1.0,0
2,731.0,N,1.0,0.0,P,40.0,286000,80,5.125,T,N,FRM,IL,PU,180,2.0,0
3,682.0,N,1.0,0.0,P,30.0,114000,80,5.500,R,N,FRM,IN,SF,180,2.0,0
4,730.0,N,1.0,0.0,P,30.0,132000,80,5.750,R,N,FRM,CO,SF,360,1.0,0


# Ex7

In [ ]:
import re
from patsy import dmatrices
from sklearn.model_selection import train_test_split

# add Loan Sequence Number back in so we can sort before splitting
model_df = mortgage_level[[
    "19_x",  # loan sequence number
    "0_x",   # credit score
    "2_x",   # first time homebuyer flag
    "6_x",   # number of units
    "5_x",   # mortgage insurance percentage
    "7_x",   # occupancy status
    "9_x",   # original DTI
    "10_x",  # original UPB
    "11_x",  # original LTV
    "12_x",  # original interest rate
    "13_x",  # channel
    "14_x",  # prepayment penalty mortgage flag
    "15_x",  # amortization type / product type
    "16_x",  # property state
    "17_x",  # property type
    "21_x",  # original loan term
    "22_x",  # number of borrowers
    "ever_delinquent"
]].copy()

model_df.columns = [
    "loan_sequence_number",
    "credit_score",
    "first_time_homebuyer_flag",
    "num_units",
    "mi_pct",
    "occupancy_status",
    "orig_dti",
    "orig_upb",
    "orig_ltv",
    "orig_interest_rate",
    "channel",
    "ppm_flag",
    "amortization_type",
    "property_state",
    "property_type",
    "orig_loan_term",
    "num_borrowers",
    "ever_delinquent"
]

# clean missing values
model_df["credit_score"] = pd.to_numeric(model_df["credit_score"], errors="coerce").replace(9999, np.nan)
model_df["first_time_homebuyer_flag"] = model_df["first_time_homebuyer_flag"].replace("9", np.nan)
model_df["num_units"] = pd.to_numeric(model_df["num_units"], errors="coerce").replace(99, np.nan)
model_df["mi_pct"] = pd.to_numeric(model_df["mi_pct"], errors="coerce").replace(999, np.nan)
model_df["occupancy_status"] = model_df["occupancy_status"].replace("9", np.nan)
model_df["orig_dti"] = pd.to_numeric(model_df["orig_dti"], errors="coerce").replace(999, np.nan)
model_df["orig_upb"] = pd.to_numeric(model_df["orig_upb"], errors="coerce")
model_df["orig_ltv"] = pd.to_numeric(model_df["orig_ltv"], errors="coerce").replace(999, np.nan)
model_df["orig_interest_rate"] = pd.to_numeric(model_df["orig_interest_rate"], errors="coerce")
model_df["channel"] = model_df["channel"].replace("9", np.nan)
model_df["ppm_flag"] = model_df["ppm_flag"].replace("9", np.nan)
model_df["amortization_type"] = model_df["amortization_type"].replace("9", np.nan)
model_df["property_state"] = model_df["property_state"].replace("", np.nan)
model_df["property_type"] = model_df["property_type"].replace("99", np.nan)
model_df["orig_loan_term"] = pd.to_numeric(model_df["orig_loan_term"], errors="coerce").replace(999, np.nan)
model_df["num_borrowers"] = pd.to_numeric(model_df["num_borrowers"], errors="coerce").replace(99, np.nan)

# drop rows with any missing values
mortgages_2004 = model_df.dropna().copy()

# clean column names for patsy
mortgages_2004.columns = [re.sub(" ", "_", c) for c in mortgages_2004.columns]
mortgages_2004.columns = [re.sub("[%/()-]", "", c) for c in mortgages_2004.columns]

# sort by Loan Sequence Number before splitting
mortgages_2004 = mortgages_2004.sort_values("loan_sequence_number").reset_index(drop=True)

# one-hot encode with patsy
y, X = dmatrices(
    "ever_delinquent ~ credit_score + C(first_time_homebuyer_flag) + C(num_units) + mi_pct + C(occupancy_status) + orig_dti + orig_upb + orig_ltv + orig_interest_rate + C(channel) + C(ppm_flag) + C(amortization_type) + C(property_state) + C(property_type) + orig_loan_term + C(num_borrowers)",
    data=mortgages_2004,
    return_type="dataframe"
)

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

results["ex7_num_obs"] = mortgages_2004.shape[0]

In [16]:
X.shape

(17052, 76)

# Ex8

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Ex9

In [18]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train.values.ravel())

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None
